# Day 09: Autoregressive Models — Tiny Char-Level GPT

Welcome to practical 9! Train a small transformer on a short text snippet.

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Fill in methods marked `# YOUR CODE HERE`.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

Autoregressive models factorize $p(x) = \prod_t p(x_t \mid x_{<t})$. A causal transformer masks future positions so each token only attends to the past.

In [ ]:
text = "to be or not to be that is the question "
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
print("Corpus length:", len(data), "Vocab:", len(chars))

### Exercise 9.1: Build training sequences

**Your job**: Sliding window of length `block_size`.

In [ ]:
block_size = 16

def get_batch(data, block_size, batch_size=32):
    # YOUR CODE HERE
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch(data, block_size)
print(xb.shape, yb.shape)

### Exercise 9.2: Tiny GPT

**Your job**: Pre-norm blocks with explicit causal self-attention, mirroring nanoGPT's `model.py`.

In [ ]:
import math

class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with a causal mask (nanoGPT-style)."""
    def __init__(self, d_model, nhead, block_size):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.qkv = nn.Linear(d_model, 3 * d_model)   # query, key, value in one matmul
        self.proj = nn.Linear(d_model, d_model)      # output projection
        # lower-triangular mask: position i may attend to j <= i only
        self.register_buffer(
            "mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, nhead, T, head_dim)
        q = q.view(B, T, self.nhead, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.nhead, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.nhead, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   # scaled dot-product
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)                  # soft dictionary lookup
        y = att @ v                                   # weighted average of past values
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    """Pre-norm Transformer block: LN -> attn -> residual, LN -> MLP -> residual."""
    def __init__(self, d_model, nhead, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, nhead, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual around attention
        x = x + self.mlp(self.ln2(x))    # residual around MLP
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, n_layers=2, block_size=16):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, nhead, block_size) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)   # token + position embeddings
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)                              # final layer norm
        return self.lm_head(x)                        # logits over the vocabulary

model = TinyGPT(len(chars), block_size=block_size).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

### Exercise 9.3: Train

**Your job**: Minimize next-token cross-entropy.

In [ ]:
for step in range(200):
    model.train()
    xb, yb = get_batch(data, block_size)
    xb, yb = xb.to(device), yb.to(device)
    # YOUR CODE HERE
    opt.zero_grad()
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, len(chars)), yb.view(-1))
    loss.backward()
    opt.step()
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss={loss.item():.3f}")

### Exercise 9.4: Generate

**Your job**: Sample characters autoregressively.

In [ ]:
@torch.no_grad()
def generate(model, prompt="t", max_new=40):
    idx = torch.tensor([[stoi[c] for c in prompt]], device=device)
    for _ in range(max_new):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_id], dim=1)
    return "".join(itos[i.item()] for i in idx[0])

print(generate(model, "to"))

## Reflection (Day 9)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (autoregressive language modeling)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?